# __Homework 4:__ Practical analysis with BioPython

For the homework, you are going to extend the code from the analysis of our FASTQ file in lectures 8 and 9.
Recall that the FASTQ file contains reads from a real sequencing run of influenza virus HA and NA genes.

---
The __actual sequences__ are as follows:

    5'-[end of HA]-AGGCGGCCGC-[16 X N barcode]-3'
or 

    5'-[end of NA]-AGGCGGCCGC-[16 X N barcode]-3'
---


__The end of NA is__ `...CACGATAGATAAATAATAGTGCACCAT`
    
__The end of HA is__ `...CCGGATTTGCATATAATGATGCACCAT`

---    

    
The __sequencing reads__ from the reverse end of the molecules (in 5'>3' orientation), so the sequencing reads are as follows:

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of HA]-3'
or

    5'-[reverse complement of 16 X N barcode]-GCGGCCGCCT-[reverse complement of the end of NA]-3'

---   
    
The reads can originate from **either** HA or NA, and that will be distinguished by the most 3' end of the read.
But in our example exercise in class, we did not distinguish among reads matching to HA and NA, as we didn't even look far enough into the read to tell the identity.

For the homework, your goal is to write code that extends the material from lectures 8 and 9 to also distinguish between HA and NA.
This homework can be completed almost entirely by re-using code from lecture 9. You will need to set up your analysis to do the following:
 1. Get the reverse complement of each read.
 2. Determine if it matches the expected pattern for HA and NA, and if so which one.
 3. If it matches, extract the barcode and add it to a dictionary to keep track of counts.
 4. Determine the number and distribution of barcodes for HA and NA separately.

Please include code to address each of the following questions. Please include code comments to explain what your code is attempting to accomplish. Don't forget to include references to the sources you used to obtain your answer, including your classmates (if you are working in groups).  

1. How many reads map to HA, and how many reads map to NA?

   Please __include at least two test cases__ showing that your HA/NA classifier works. You can copy the `run_tests` helper from lecture 9, or use `assert` statements - either is fine. These test cases are part of the points for this question.


In [65]:
import Bio.SeqIO
import Bio.Seq
import re

# SeqIO.parse returns reads one at a time; list() pulls them all into memory
seqreads = list(Bio.SeqIO.parse("barcodes_R1.fastq", format="fastq"))

print(f"Found {len(seqreads)} sequencing reads.")



def read_barcode(seqread):
    """Identify a barcode with a known upstream sequence.

    Args:
        seqread: Nucleotide sequence read (a Seq object or a string) matching
            in reverse orientation.
        bclen: Length of the barcode.
        upstream: Sequence expected immediately upstream of the barcode (forward orientation).

    Returns:
        The barcode in the forward orientation, as an upper-case string, or
        None if the read does not match the expected design.

    Example:
        >>> read_barcode(Bio.Seq.Seq("TTTTTTTTTTTTTTTTGCGGCCGCCT"), bclen=16)
        'AAAAAAAAAAAAAAAA'
    """

    seq_rc = Bio.Seq.Seq(str(seqread)).reverse_complement() 
    # "Take whatever was passed as "seqread", convert it to a string, and get the reverse complement."
    pattern_na = re.compile(r"CACGATAGATAAATAATAGTGCACCAT")

    pattern_ha = re.compile(r"CCGGATTTGCATATAATGATGCACCAT")


    pattern = re.compile(r"AGGCGGCCGC(?P<barcode>[ACTGN]{16})$")
    # "Find the upstream sequence. Then find a length-16 sequence (called 'barcode') containing only A, C, G, T, or N."

    match_na = pattern_na.search(str(seq_rc))

    match_ha = pattern_ha.search(str(seq_rc))



    match = pattern.search(str(seq_rc))
    # "Search the reverse complement string for our pattern."

    

    if match and (match_na or match_ha):
        return str(match.group("barcode")).upper(), match_na, match_ha
    # "If there is a match, get the 'barcode' part of the sequence and return it."

    # "If we reach this point, we did not find a valid match. Return None."

    return None, match_na, match_ha

Found 10000 sequencing reads.


In [63]:
ha_sequence = (
    "CCGGATTTGCATATAATGATGCACCAT"
    + "AGGCGGCCGC"
    + "ACGTACGTACGTACGT"
)

ha_test_read = Bio.Seq.Seq(ha_sequence).reverse_complement()


# Create a known NA sequence and put it in sequencing-read orientation
na_sequence = (
    "CACGATAGATAAATAATAGTGCACCAT"
    + "AGGCGGCCGC"
    + "ACGTACGTACGTACGT"
)

na_test_read = Bio.Seq.Seq(na_sequence).reverse_complement()

print(read_barcode(ha_test_read))

Testing raw seqquence : ACGTACGTACGTACGTGCGGCCGCCTATGGTGCATCATTATATGCAAATCCGG
('ACGTACGTACGTACGT', None, <re.Match object; span=(0, 27), match='CCGGATTTGCATATAATGATGCACCAT'>)


In [77]:
barcode_counts = {"NA" : {}, "HA" : {}, "None" : {}}
unmatched = []
count_na = 0
count_ha = 0
count_neither = 0
for read in seqreads:
    print(f"Read: {read.id} : {read.seq}")

    barcode, match_na, match_ha = read_barcode(read.seq)

    if match_na:
        try: barcode_counts["NA"][barcode]+=1
        except: barcode_counts["NA"][barcode]=1
        count_na+=1
    elif match_ha:
        try: barcode_counts["HA"][barcode]+=1
        except: barcode_counts["HA"][barcode]=1
        count_ha+=1
    else:
        try: barcode_counts["None"][barcode]+=1
        except: barcode_counts["None"][barcode]=1
        count_neither+=1
        unmatched.append(read.id)

    print(f"Barcode: {barcode}, Match NA: {match_na}, Match HA: {match_ha}")

    

print(barcode_counts)
print(f"HA: {count_ha}, NA:{count_na}, neither:{count_neither}")



Read: HISEQ:621:HMJGNBCX2:1:1101:1797:2150 : GCTTAAGTTATTTAGTGCGGCCGCCTATGGTGCACTATTATTTATCTATCGTGAAAGGGAGTTCTGCTCCATCAGGCCAAGATCGGAAGAGCACACGTCTGAACTCCAGTCACATGTCAGAATCTCGTATGCCGTCTTCTGCTTGAAAAAAAAAAAAAAGCTGAAATTAATAATTTTGAAACCAGTTTTGTAAACGCAGCACTAAAATGAAGGCATGCAACGACGATGTTTATTGACACGGAATAGCAGA
Barcode: ACTAAATAACTTAAGC, Match NA: <re.Match object; span=(197, 224), match='CACGATAGATAAATAATAGTGCACCAT'>, Match HA: None
Read: HISEQ:621:HMJGNBCX2:1:1101:9239:2123 : CTTCCTGGTCACGGTTGCGGCCGCCTATGGTGCATCATTATATGCAAATCCGGCATTGCAAGGAGCCGTTGGAACACATAGATCGGAAGAGCACACGTCTGAACTCCAGTCACATGTCAGAATCTCGTATGCCGTCTTCTGCTTGAAAAAAAAAAAAAAAACCTAAATAGGTATCATAAAGACACCATCAACAGCTTAGTATACAACCCCATGCATTAAACAAATACATCCCTAATGACAGAGTGTACGA
Barcode: AACCGTGACCAGGAAG, Match NA: None, Match HA: <re.Match object; span=(197, 224), match='CCGGATTTGCATATAATGATGCACCAT'>
Read: HISEQ:621:HMJGNBCX2:1:1101:11925:2202 : CACGGCTATTACCCCGCGGCCGCCTATGGTGCACTATTATTTATCTATCGTGAAAGGGAGTTCTGCTCCATCAGGCCAAGATCGGAAGAGCACACGTCTGAACTCCAGTCAC

In [75]:
print(read_barcode("CCGCGGGGATGCGACGGCCGCCTATGGTGCATCATTATATGCAAATCCGGCATTGCAAGGAGCCGTTTGAACACATAGATCGGAAGGGCACAAGTCTGAACTCCGGTCACATGTCAGAATCTCGTATGCCGTCTTCCGCGTGTAAATACAAATCACTGGGTAAGGGCATTCACCAGATTTTTGGAGCAGCCTTCAAATCACAGTTTGGAGGAATGTCCTGGTTCTCACAGATCCTCATAGGCACGCAGCT"))

(None, None, <re.Match object; span=(200, 227), match='CCGGATTTGCATATAATGATGCACCAT'>)


Testing raw seqquence : TTTTTTTTTTTTTTTTGCGGCCGCCT
RC : AGGCGGCCGCAAAAAAAAAAAAAAAA
Match: <re.Match object; span=(0, 26), match='AGGCGGCCGCAAAAAAAAAAAAAAAA'>
None
Testing raw seqquence : TTTTTTTTTTTTTTTTGCGGCCGCCT
RC : AGGCGGCCGCAAAAAAAAAAAAAAAA
Match: <re.Match object; span=(0, 26), match='AGGCGGCCGCAAAAAAAAAAAAAAAA'>


AssertionError: 

2. How many HA sequences did not have a valid barcode? Also anwer the same question for NA.

In [60]:
print(barcode_counts["HA"][None])
print(barcode_counts["NA"][None])
print(len(barcode_counts["HA"].keys())-1)
print(len(barcode_counts["NA"].keys())-1)

163
212
281
138


3. What is the HA barcode with the most counts (and how many counts)? Also answer the same question for NA.

    _Hint: you will need to find the key associated with the maximum value in your dictionary. There are many ways to do this._

In [61]:
# your code here...

# get the most common barcode for Ha and NA that isn't None

na_barcodes = {k: v for k, v in barcode_counts["NA"].items() if k is not None}
ha_barcodes = {k: v for k, v in barcode_counts["HA"].items() if k is not None}

max_na_barcode = max(na_barcodes, key=na_barcodes.get)
max_ha_barcode = max(ha_barcodes, key=ha_barcodes.get)

print(f"Most common NA barcode: {max_na_barcode} ({na_barcodes[max_na_barcode]} reads)")
print(f"Most common HA barcode: {max_ha_barcode} ({ha_barcodes[max_ha_barcode]} reads)")

Most common NA barcode: ACCAGTTCTCCCCGGG (152 reads)
Most common HA barcode: CCCGACCCGACATTAA (155 reads)
